# E011 — How much of the model's skill ranks ads?

**Hypotheses:** H1, H11 ([docs/hypotheses.md](../../docs/hypotheses.md))

**Result:** Only 3.4 % of the model's skill ranks ads: replacing each candidate's pCTR with its opportunity's mean costs 0.0018 log loss [0.0010, 0.0030] (NE 0.8842 -> 0.8882); the rest predicts whether a moment is clickable at all, which the ranker cannot use.

**Question:** NE rewards predicting *whether* a moment gets a click. The ranker only uses *differences between the candidates* of one moment. How much of the model's skill is the second kind?

**Method:** An opportunity is a test impression with its reconstructed candidate set (the off-policy evaluation's publisher x hour cell; every candidate scored in that impression's context). On the logged ad, compare the shipped pCTR with the mean pCTR of the opportunity's candidates, which keeps the context signal and removes every difference between ads. Paired hour-block bootstrap of the log-loss gap. Suggested by the external reviewer's flattening diagnostic (NE 0.8896 -> 0.8940 on their rows).

In [1]:
from charade.analysis.experiment import setup

ctx = setup()
f"mode: {'smoke (fixture, tiny model)' if ctx.smoke else 'full data'}"

'mode: full data'

In [2]:
import json

from charade.analysis import opportunity
from charade.analysis.experiment import ensure_bundle

ensure_bundle(ctx)
report = opportunity.run(ctx.settings, ctx.data_dir, ctx.reports / "models" / "opportunity.json")
print(json.dumps(report, indent=2))

{
  "opportunities": 102874,
  "mean_candidates": 6.998085036063534,
  "model": {
    "ne": 0.8842318166796227,
    "auc": 0.7393005129761667
  },
  "flattened": {
    "ne": 0.8881888633152322,
    "auc": 0.7357722509522469
  },
  "logloss_cost_of_flattening": {
    "delta": 0.0018183087329393664,
    "ci_low": 0.0009815574527979242,
    "ci_high": 0.0029739452217340773
  },
  "share_of_skill_that_ranks_ads": 0.03418077853617801
}


## Reading
- The headline NE (0.8855) is mostly *context* skill: which publisher, device and user are likely to click. E009 agrees: `app_id`, `site_id` and `site_domain` are the three most needed features. Within one request that signal is identical for every candidate and cancels out.
- The part that changes which ad is served is small but real (CI excludes zero), consistent with the modest offline policy lift (+1.46 pp, [04](../../docs/04-ranking-policy.md)).
- Avazu ads are anonymous ids, so there is little to tell them apart. Simula's value is exactly this within-opportunity part (which ad fits this conversation), which needs ad content and conversation features ([08](../../docs/08-next-steps.md)).
- Evaluation consequence: report a within-opportunity metric next to NE, because that is what the ranker consumes.

## Conclusion
Only 3.4 % of the model's skill ranks ads: replacing each candidate's pCTR with its opportunity's mean costs 0.0018 log loss [0.0010, 0.0030] (NE 0.8842 -> 0.8882); the rest predicts whether a moment is clickable at all, which the ranker cannot use.

**Decision:** No model change. The flattening gap is reported next to NE in the summary, and next steps prioritise ad-content and conversation features, which act within opportunities.